# CHIIR 2027 proceedings procedure

Use this notebook during the proceedings period, from the acceptance decisions to the front matter. Do the steps in sequence. Each step gives its instructions and contains its settings.

Start Jupyter in the root folder of the repository. If you start it in a different folder, Python cannot find the `proceedings` package.

**WARNING:** The cell outputs contain the names and emails of authors. Do not commit them. The first cell installs the `nbstripout` Git filter in this repository if it is not installed. This filter removes the outputs before each commit.

In [ ]:
YEAR = 2027
NAME = "CHIIR"

import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

from proceedings import (CategoryRule, ConferenceConfig, PaperType, check_erights_csv,
                         check_xml_against_source, export_to_erights_csv, export_to_xml,
                         heading_previews, load_affiliation_mapping, load_easychair_data)

# Show only the errors from the loader. The notebook shows the other results.
logging.basicConfig(level=logging.ERROR, format="%(message)s", stream=sys.stdout, force=True)

# Install the nbstripout Git filter, so that outputs (author data) are never committed
import subprocess
import proceedings
REPO_ROOT = Path(proceedings.__file__).resolve().parent.parent
nbstripout = [sys.executable, "-m", "nbstripout"]
if subprocess.run([*nbstripout, "--is-installed"], cwd=REPO_ROOT, capture_output=True).returncode != 0:
    result = subprocess.run([*nbstripout, "--install"], cwd=REPO_ROOT, capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Installed the nbstripout Git filter in {REPO_ROOT}. Git now removes notebook outputs before each commit.")
    else:
        print("WARNING: Could not install the nbstripout Git filter. Do not commit this notebook with outputs.")
        print(result.stderr.strip())

## 1. Download the data from EasyChair

Do this step when the decisions are final. Do it again each time that you correct data in EasyChair.

Use one of these two downloads. They give the same result.

| Download | Location in EasyChair | File names |
|---|---|---|
| Excel workbook (your EasyChair licence must allow it) | **Conference → Data download/Export → Excel** | `export.xlsx` |
| CSV pair | **Submissions → Download**: *All submissions (CSV)* and *Authors (CSV)* | `submission.csv`, `author.csv` |

**WARNING:** Do not save the files in this repository. The files contain personal data.

To try the notebook with invented data, do these steps:

1. Run `python tests/fixtures/make_fixtures.py ~/CHIIR2027-demo`.
2. Set `DATA_DIR` to that folder.

In [ ]:
DATA_DIR = Path.home() / "CHIIR2027-proceedings"

EASYCHAIR_EXPORT = DATA_DIR / "export.xlsx"
# EASYCHAIR_EXPORT = (DATA_DIR / "submission.csv", DATA_DIR / "author.csv")

assert REPO_ROOT not in [DATA_DIR.resolve(), *DATA_DIR.resolve().parents], "Put the EasyChair data in a folder outside the repository"

## 2. Set the accepted decisions and the paper types

**Accepted decisions:** Write the EasyChair decision labels exactly. Upper case and lower case are not important. The cell shows a count for each decision. Examine this count. The notebook does not use papers with other decisions.

**Paper types:** These are the names that ACM shows in eRights. Write them in the sequence of the proceedings. The prefix is the start of the event tracking number. For example, submission 42 of type `fp` becomes `fp42`. Before you upload, make sure that the ACM proceedings coordinator agrees with the names.

**Category rules:** The notebook examines the rules from the top to the bottom. The first rule that agrees with the paper gives the paper type.

At CHIIR, authors of the main track select a category in the `(Category) …` form field. The DC, the tutorials and the workshops have their own tracks.

If you add a track or a category to EasyChair, add a rule here. If an accepted paper agrees with no rule, step 3 shows an error.

In [ ]:
ACCEPTED_DECISIONS = ["accept", "probably accept"]

PAPER_TYPES = [
    PaperType(name="Full Paper",          prefix="fp", heading="Full Papers",         file_stem="Full-Papers"),
    PaperType(name="Short Paper",         prefix="sp", heading="Short Papers",        file_stem="Short-Papers"),
    PaperType(name="Demonstration Paper", prefix="dp", heading="Demonstrations",      file_stem="Demonstrations"),
    PaperType(name="Doctoral Consortium", prefix="dc", heading="Doctoral Consortium", file_stem="Doctoral-Consortium"),
    PaperType(name="Tutorial",            prefix="tp", heading="Tutorials",           file_stem="Tutorials"),
    PaperType(name="Workshop",            prefix="wk", heading="Workshops",           file_stem="Workshops"),
]

CATEGORY_RULES = [
    CategoryRule(field="form fields", contains=["(category) full paper", "(category) resource paper",
                                                "(category) perspective"],      paper_type="Full Paper"),
    CategoryRule(field="form fields", contains=["(category) demonstration"],   paper_type="Demonstration Paper"),
    CategoryRule(field="form fields", contains=["(category) short paper"],     paper_type="Short Paper"),
    CategoryRule(field="track name",  contains_all=["chiir", "dc"],            paper_type="Doctoral Consortium"),
    CategoryRule(field="track name",  contains_all=["chiir", "short"],         paper_type="Short Paper"),
    CategoryRule(field="track name",  contains=["tutorial"],                   paper_type="Tutorial"),
    CategoryRule(field="track name",  contains=["workshop"],                   paper_type="Workshop"),
    CategoryRule(field="track name",  contains_all=["chiir", "pc"],            paper_type="Full Paper"),
]

config = ConferenceConfig(name=NAME, year=YEAR, accepted_decisions=ACCEPTED_DECISIONS,
                          paper_types=PAPER_TYPES, category_rules=CATEGORY_RULES)


def load(affiliation_mapping=None, **options):
    return load_easychair_data(EASYCHAIR_EXPORT, config, affiliation_mapping=affiliation_mapping, **options)


export = load()

display(pd.Series(export.decision_counts, name="submissions").to_frame()
        .assign(included=lambda df: [config.is_accepted(d) for d in df.index]))
display(pd.DataFrame([(t.section_name, t.original_name, t.paper_count) for t in export.tracks],
                     columns=["paper type", "EasyChair track(s)", "papers"]))

## 3. Correct the data before you send it to ACM

The previous CHIIR proceedings chairs give this advice:

- **TAPS uses the eRights data for the paper heading.** TAPS does not use the affiliations, emails or addresses in the camera-ready source. If an author writes a department, a street address or two institutions in one field, TAPS prints that text. To correct the heading later, ACM must **remove the paper from TAPS and remove its eRights form**. Then the authors must complete the form again. It is much easier to correct the data now.
- **Only one contact author receives the ACM emails.** Authors often mark more than one author as corresponding in EasyChair. Previously, the last marked author received the emails. This author was usually a senior author with little time. Now the contact author is the **first** marked author who has an email. The table shows each paper that has more than one marked author. Make sure that the contact author is correct.

The affiliation map (`affiliations_mapping.json`) changes affiliation names to a standard name, for example "Adobe Inc." → "Adobe". The default is off, because the map changes the text of the authors. Use the map only if you want the same name for each institution. If you use it, examine the `affiliation_canonicalised` notices.

In [ ]:
AFFILIATION_MAPPING = None
# AFFILIATION_MAPPING = load_affiliation_mapping("affiliations_mapping.json")

export = load(AFFILIATION_MAPPING)

NEEDS_ACTION = ["uncategorised", "no_authors", "author_validation_error", "paper_validation_error",
                "affiliation_too_detailed", "multiple_corresponding", "invalid_email",
                "contact_author_priority_2", "contact_author_priority_3", "missing_email",
                "missing_affiliation", "author_order_mismatch", "name_consistency", "missing_approval_date"]

issues = pd.DataFrame([i.model_dump(exclude={"details"}) for i in export.validation_issues])
if issues.empty:
    print("No problems found")
else:
    issues["category"] = pd.Categorical(issues["category"], NEEDS_ACTION + sorted(set(issues["category"]) - set(NEEDS_ACTION)))
    with pd.option_context("display.max_rows", None, "display.max_colwidth", 120):
        display(issues[issues["category"].isin(NEEDS_ACTION)]
                .sort_values(["category", "paper_id"])[["category", "severity", "paper_id", "author_name", "message"]])
    print("Other notices (usually correct):", issues[~issues["category"].isin(NEEDS_ACTION)]["category"].value_counts().to_dict())

Correct each problem in EasyChair. Then download the export again (step 1) and run steps 2 and 3 again.

| Category | Action |
|---|---|
| `uncategorised` | Add a category rule in step 2. Alternatively, correct the form field or the track of the paper. |
| `affiliation_too_detailed` | Change the affiliation of the author to the name of the institution only. Remove the department, the address and the country. The country has its own field. |
| `multiple_corresponding` | This is usually correct. The message gives the name of the author who will receive the ACM emails. If this author is not correct, remove the corresponding mark from the other authors. |
| `invalid_email`, `missing_email`, `contact_author_priority_*` | Get a correct email for the contact author. Without an email, nobody receives the eRights link. |
| `missing_affiliation` | Ask the authors for the affiliation. |
| `missing_approval_date` | No action now. You set the dates in step 5. |
| `author_order_mismatch`, `name_consistency` | A name in the author list of the submission is different from the name in the author record. This is usually a typing error. Make the two names the same. |

## 4. Ask the authors to examine their headings

Send each contact author the block for their paper (below the email template). Do this before you upload data to eRights.

Be prepared to send reminders more than one time. The previous chairs had one or two authors who did not obey the instructions.

**Email template**

> **Subject:** CHIIR 2027 – examine your author data (paper #ID)
>
> Dear NAME,
>
> Congratulations again on the acceptance of your paper. You are the contact author. Thus, ACM will send you two emails in the next weeks:
>
> 1. A link to the **eRights** copyright form
> 2. An invitation to upload your final version to **TAPS**
>
> **Important:** TAPS makes the author block of the published paper from the eRights data. TAPS does **not** use the author block in your LaTeX or Word source. The block below shows the data that we will send to ACM. Examine the names, the author sequence, the affiliations and the emails. Send us the corrections before DATE.
>
> - For the affiliation, write only the name of the institution, for example "University of Glasgow". Do not write the department, the street address or the postcode. The country has its own field.
> - If you change the eRights form, use the same format.
> - If the heading is not correct after you submit the form, we must remove your paper from TAPS and eRights. Then you must complete the form again.
> - Only the contact author receives the ACM emails. Tell your co-authors about these emails.
>
> PREVIEW

In [ ]:
for preview in heading_previews(export):
    print(f"To: {preview['contact_name']} <{preview['contact_email']}>")
    print(preview["text"])
    print("-" * 80)

## 5. Make the eRights upload

Set these values:

- **ACM event ID:** This number is in the eRights confirmation email from ACM, for example `3726522`.
- **Proceeding ID:** The XML header contains this value. Use the value from the ACM coordinator.
- **Dates:** Use the ACM format `DD-MON-YYYY`. Set the submission date to the submission deadline. Set the approval date to the notification date. If you do not set the dates, the notebook uses the dates from EasyChair. The "last updated" time in EasyChair is *not* the approval date.

The cell writes these files:

- One CSV file for each paper type, and `CHIIR-2027-All-Papers.csv`. These files use the ACM Enhanced CSV format with 36 columns, the same as for CHIIR 2026.
- One ACM XML file with the same data.

Upload the format that ACM asks for. The cell compares the two formats with the EasyChair export.

In [ ]:
EVENT_ID = "PLACEHOLDER"           # from the eRights confirmation email
PROCEEDING_ID = f"{YEAR}-{NAME}"
SUBMISSION_DATE = "15-OCT-2026"    # DD-MON-YYYY
APPROVAL_DATE = "01-DEC-2026"
OUT_DIR = DATA_DIR / "output"

export = load(AFFILIATION_MAPPING, proceeding_id=PROCEEDING_ID,
              submission_date_override=SUBMISSION_DATE, approval_date_override=APPROVAL_DATE)

OUT_DIR.mkdir(parents=True, exist_ok=True)
xml_file = OUT_DIR / f"{NAME.lower()}{YEAR}.xml"
export_to_xml(export, str(xml_file))
problems = check_xml_against_source(EASYCHAIR_EXPORT, config, xml_file, affiliation_mapping=AFFILIATION_MAPPING)

if EVENT_ID.startswith("PLACEHOLDER"):
    print("Set EVENT_ID. Then run this cell again to write the eRights CSV files.")
else:
    csv_files = export_to_erights_csv(export, OUT_DIR, EVENT_ID, config)
    problems += [p for path in csv_files.values() for p in check_erights_csv(path)]
    display(pd.DataFrame([(k, v.name) for k, v in csv_files.items()], columns=["paper type", "file"]))

print(f"XML: {xml_file}")
print("Cross-check:", "OK" if not problems else "")
for problem in problems:
    print("  ✗", problem)

This cell is optional. It does more analysis of the XML:

- It makes sure that each paper has one contact author.
- It finds affiliations with similar spellings.
- It finds authors with two names.
- It shows the most frequent countries and institutions.

In [ ]:
from validate_acm_xml import validate_xml_file

validate_xml_file(str(xml_file), top_k=5);

Upload the CSV files (or the XML file) for your event at <https://erightsreview.acm.org>. Then ACM sends an email to each contact author. This email asks the author to complete the eRights form.

**NOTE:** TAPS does not start to process a paper until the authors complete the eRights form.

After eRights, the authors receive TAPS invitations. They upload their LaTeX or Word source. During this period, do these tasks:

- Send reminders to the authors who miss the TAPS deadline.
- Examine and approve the papers that the TAPS quality checks identify.

You can now make the list of accepted papers for the website:

In [ ]:
from papers_to_docx import create_word_doc, extract_papers_from_xml

create_word_doc(extract_papers_from_xml(str(xml_file)), str(OUT_DIR / "accepted-papers.docx"), config)

## 6. Send one paper again (only if a heading is not correct)

Do this step only if a heading is not correct after the paper is in TAPS.

1. Ask ACM to **remove the paper from TAPS and remove its eRights form**. The author cannot change the data before ACM does this.
2. Correct the data in EasyChair.
3. Download the export again (step 1).
4. Run steps 2 and 3 again.
5. Set `PAPER_ID` below and run the cell. The cell writes the CSV and XML files for that paper only.
6. Upload the files again.

In [ ]:
PAPER_ID = None   # EasyChair submission number, for example 42

if PAPER_ID is not None:
    reissue = load(AFFILIATION_MAPPING, paper_id_filter=PAPER_ID, proceeding_id=PROCEEDING_ID,
                   submission_date_override=SUBMISSION_DATE, approval_date_override=APPROVAL_DATE)
    reissue_dir = OUT_DIR / f"reissue-{PAPER_ID}"
    reissue_dir.mkdir(parents=True, exist_ok=True)
    export_to_xml(reissue, str(reissue_dir / f"{NAME.lower()}{YEAR}-{PAPER_ID}.xml"))
    if not EVENT_ID.startswith("PLACEHOLDER"):
        export_to_erights_csv(reissue, reissue_dir, EVENT_ID, config)
    print(next(p["text"] for p in heading_previews(reissue)))
    print(f"\nFiles are in {reissue_dir}")

## 7. Table of contents (after TAPS approves all papers)

1. In the TAPS admin panel, select **Export Metadata**.
2. Save the file as `ExportMetadata.xlsx` in `DATA_DIR`.
3. Examine the file:
   - Make sure that `ORDER ID` gives the correct sequence for the proceedings. If it does not, change the sequence in TAPS and export the file again.
   - Make sure that `PDFPAGES` is final. TAPS can show a temporary page count before it completes a paper.
4. Run the cell.

The page numbers start at 1 and continue through all papers.

In [ ]:
TAPS_EXPORT = DATA_DIR / "ExportMetadata.xlsx"

from generate_toc import generate_toc

if TAPS_EXPORT.exists():
    toc_text, toc = generate_toc(str(TAPS_EXPORT), str(OUT_DIR / f"{NAME.lower()}{YEAR}_toc"))
    display(toc)
else:
    print(f"Save the TAPS metadata export as {TAPS_EXPORT}. Then run this cell again.")

Examine the result:

- Make sure that the paper types are in the correct sequence.
- Make sure that the page counts are correct. Full papers usually have 10 to 12 pages. Short papers usually have 4 pages.
- Make sure that the author names contain no HTML entities and no incorrect characters.

## 8. Front matter

### Committee lists

Ask each track chair for the PC and SPC spreadsheets. Put the spreadsheets in `DATA_DIR / "referees"`. The spreadsheets can be EasyChair or OpenReview exports. They contain columns such as first name, last name, email, affiliation and role.

The scan compares the files. It shows duplicates and differences *between* files, for example:

- The same person with a different affiliation or email
- Typing errors in names

Then it merges all data into `referees.xlsx`. This file has one sheet for each track and role.

Set `REFEREE_LAYOUT` for the files of this year:

- `file_tracks` gives the track of each file. If all persons in a file have the same role, it also gives the role, for example `"PC"` or `"SPC"`.
- `file_roles` gives the role for files that have no role column.
- `track_order` gives the sequence of the sheets.

In [ ]:
REFEREE_DIR = DATA_DIR / "referees"
REFEREE_LAYOUT = {
    "file_tracks": {
        # "full_papers_pc.xlsx": ["Full Papers", "PC"],
        # "full_papers_spc.xlsx": ["Full Papers", "SPC"],
    },
    "file_roles": {},
    "track_order": ["Full Papers", "Short Papers", "Demonstrations", "Doctoral Consortium"],
}

import scan_referees

if REFEREE_DIR.is_dir():
    scan_referees.configure(**REFEREE_LAYOUT)
    scan_referees.main(["--input", str(REFEREE_DIR), "--output", str(OUT_DIR / "referees.xlsx"),
                        "--report", str(OUT_DIR / "referees-report.html")])
else:
    print(f"Put the committee spreadsheets in {REFEREE_DIR}. Then run this cell again.")

1. Correct the differences in the source spreadsheets.
2. Run the scan again.
3. In `COMMITTEE_SECTIONS`, write the sheets of `referees.xlsx` for the front matter, with their headings. Write them in the sequence of the front matter.
4. Run the cell.

The output uses the ACM fonts: Linux Biolinum for the headings and Linux Libertine for the names. After the cell writes the file, it compares the file with the spreadsheet.

In [ ]:
COMMITTEE_SECTIONS = [
    # "<sheet in referees.xlsx>=<heading>"
    # "Full Papers - SPC=Senior Program Committee:",
    # "Full Papers - PC=Program Committee:",
]

from referees_to_docx import load_sections, write_committee_docx

if COMMITTEE_SECTIONS and (OUT_DIR / "referees.xlsx").exists():
    write_committee_docx(load_sections(str(OUT_DIR / "referees.xlsx"), COMMITTEE_SECTIONS),
                         str(OUT_DIR / "committee.docx"))
else:
    print("Write COMMITTEE_SECTIONS when referees.xlsx is correct. Then run this cell again.")

### Make and submit the front matter

Start with `templates/CHIIR26FrontMatter_Example.docx`. Change it for this year:

| Section | Content |
|---|---|
| Cover page | Conference name, year, dates, location, ISBN |
| Welcome messages | The messages of the general chairs and the PC chairs |
| Organising committee | Names and affiliations for each role |
| Programme committee | Copy from `committee.docx` |
| Sponsors | Logos and names |
| Table of contents | Copy from the `_toc.txt` file of step 7. Apply the `TOC Entry` paragraph style. |

**NOTE:** The page numbers in the table of contents are the page numbers of the proceedings PDF, not of the Word document. Thus, remove the Word page numbers from this section.

Ask the ACM proceedings coordinator for the necessary format (PDF or `.docx`) and the deadline.

## Optional: save the settings for the command line

This cell writes the conference settings of this year to a JSON file. Use this file with `easychair_export.py --config` to do the export again without the notebook.

In [ ]:
config_file = OUT_DIR / f"{NAME.lower()}{YEAR}.json"
config.to_json(config_file)
inputs = " ".join(map(str, EASYCHAIR_EXPORT)) if isinstance(EASYCHAIR_EXPORT, tuple) else str(EASYCHAIR_EXPORT)
print(f"Saved {config_file}\n")
print(f"python easychair_export.py --config {config_file} --input {inputs} --format csv "
      f"--event_id {EVENT_ID} --submission_date {SUBMISSION_DATE} --approval_date {APPROVAL_DATE} --output {OUT_DIR}")